## Market Maker: run the following cell, then press the button "Run MarketMaker"

In [ ]:
## Setup

from eth_account import Account
from web3 import Web3
import ipywidgets as widgets
import time
import logging
from scripts import config as c
from scripts import web3_client_handler as web3Client

w3 = web3Client.get_w3_client()

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")
logger = logging.getLogger(__name__)

## helper functions


def _buy(security_address: str, price: int):
    print("Placing buy order: " + security_address + " " + str(price))
    # _do_contract_tx(orderbook, "buy", (security_address, 1, price, account.address, account.address, tbd_nordea.address), c.PK_BROKER2.address, c.PK_BROKER2.key)
    orderbook.do_contract_tx(
        "buy",
        (
            security_address,
            1,
            price,
            account.address,
            account.address,
            tbd_nordea.contract_address,
        ),
        c.PK_BROKER2.key,
        logger,
    )


def _sell(security_address: str, price: int):
    print("Placing sell order: " + security_address + " " + str(price))
    # _do_contract_tx(orderbook, "sell", (security_address, 1, price, account.address, account.address, tbd_nordea.address), c.PK_BROKER2.address, c.PK_BROKER2.key)
    orderbook.do_contract_tx(
        "sell",
        (
            security_address,
            1,
            price,
            account.address,
            account.address,
            tbd_nordea.contract_address,
        ),
        c.PK_BROKER2.key,
        logger,
    )


## setup
address = c.registry_contract_address
with open("contracts/GlobalRegistry.sol/GlobalRegistry.abi", "r") as f:
    abi = "".join(f.read().splitlines())
global_registry = w3.eth.contract(address=address, abi=abi)

orderbook = web3Client.Web3ClientHandler(w3, global_registry, "Order Book", "OrderBook")

tbd_nordea = web3Client.Web3ClientHandler(w3, global_registry, "TBD Nordea", "Tbd")

## MarketMaker address
account = c.PK_MARKET_MAKER

# Print out the private key and the corresponding Ethereum address
print("Private Key:", account.key.hex())
print("Ethereum Address:", account.address)

# Add new account to the bank's allowlist. Use either PK_NORDEA or PK_DNB, depending on which bank you want to sign up with
tbd_nordea.do_contract_tx("add", [account.address], c.PK_NORDEA.key, logger)

# Init funds
tbd_nordea.do_contract_tx(
    "mint", [account.address, 1000000000], c.PK_NORDEA.key, logger
)

## MarketMaker core function


def _run(b):
    i = 0
    captured_prices = {}

    while True:
        print("Market Maker round: " + str(i))
        i = i + 1
        time.sleep(2)

        buy_orders = orderbook.contract.functions.getAllBuyOrders().call(
            {"from": c.PK_CSD.address}
        )
        sell_orders = orderbook.contract.functions.getAllSellOrders().call(
            {"from": c.PK_CSD.address}
        )

        securities = orderbook.get_all_isins()
        for security_isin in securities:

            security_contract, security_address = orderbook.get_stock_token_contract(
                security_isin
            )
            own_balance = security_contract.functions.balanceOf(account.address).call()

            bid_prices = []
            ask_prices = []

            if security_address not in captured_prices:
                print(
                    "New security found: "
                    + security_address
                    + " waiting 10s, until initial order placement is done (otherwise, due to a web3 conflict, the issuance process will fail)"
                )
                time.sleep(10)
                print("Added new security: " + security_address)
                captured_prices[security_address] = {}
                captured_prices[security_address]["bid"] = []
                captured_prices[security_address]["ask"] = []
                # add new account to allowlist of isin
                orderbook.do_contract_tx(
                    "add", [account.address], c.PK_CSD.key, logger, security_contract
                )

            # update statistics
            for order in buy_orders:
                # order = id, address broker, investorSecAddr, secContrAddr, amount, price, investorTbdAddr, tbdContrAddr
                if order[3] == security_address:
                    captured_prices[security_address]["bid"].append(order[5])
                    bid_prices.append(order[5])

            for order in sell_orders:
                # order = id, address broker, investorSecAddr, secContrAddr, amount, price, investorTbdAddr, tbdContrAddr
                if order[3] == security_address:
                    captured_prices[security_address]["ask"].append(order[5])
                    ask_prices.append(order[5])

            ## create new orders ##

            # there are buy and sell orders, if the spread is larger than 1: place a buy order / sell order to reduce the spread
            if len(bid_prices) > 0 and len(ask_prices) > 0:
                highest_bid = max(bid_prices)
                smallest_ask = min(ask_prices)
                if smallest_ask - highest_bid > 1:
                    if own_balance == 0:
                        buy_price = highest_bid + 1
                        _buy(security_address, buy_price)
                    else:
                        sell_price = smallest_ask - 1
                        _sell(security_address, sell_price)
                else:
                    continue

            # there are only buy orders and we have the security, place a sell order slightly above the highest bid
            elif len(bid_prices) > 0 and len(ask_prices) == 0 and own_balance > 0:
                sell_price = max(bid_prices) + 1
                _sell(security_address, sell_price)

            # there are only buy orders and we do not have the security, do nothing
            elif len(bid_prices) > 0 and len(ask_prices) == 0 and own_balance == 0:
                continue

            # there are no buy orders and we don't have the security, buy it cheaply
            elif len(ask_prices) > 0 and own_balance == 0:
                buy_price = min(ask_prices)
                _buy(security_address, buy_price)

            # there are only sell orders and we have the security, do nothing
            elif len(ask_prices) > 0 and own_balance > 0:
                continue

            # there are no buy or sell orders, but we have the security at least twice, create a historical sell order
            elif own_balance > 1 and len(captured_prices[security_address]["bid"]) > 0:
                sell_price = max(captured_prices[security_address]["bid"])
                _sell(security_address, sell_price)

            # there are no buy or sell orders and we don't have the security, create a historical buy order
            elif own_balance == 0 and len(captured_prices[security_address]["ask"]) > 0:
                buy_price = min(captured_prices[security_address]["ask"])
                _buy(security_address, buy_price)

            # all other cases: do nothing
            continue

In [ ]:
## Create Start button
button = widgets.Button(description="Run MarketMaker")
output = widgets.Output()
button.on_click(_run)
widgets.HBox([button, output])